# 03 — Feature Engineering
Builds the feature matrix that the ML models will train on:
- **Text features**: TF-IDF on headline + text
- **FinBERT embeddings**: 768-d CLS vector
- **Lexical sentiment**: VADER + custom banking lexicon
- **Temporal features**: day-of-week, lag indicators
- **Price labels**: stock movement 1-day and 5-day forward

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
from tqdm.notebook import tqdm
import pickle, warnings
warnings.filterwarnings('ignore')

ROOT = Path('..')
PROC = ROOT / 'data' / 'processed'

df = pd.read_csv(PROC / 'all_banking_news.csv', parse_dates=['date'])
print(f'Loaded {len(df):,} articles')
df.head(3)

Loaded 13,649 articles


,ticker,date,source,headline,text
0,SBIN,2022-02-07,moneycontrol,What should investors do with SBI after Q3 res...,NaN
1,SBIN,2022-02-08,moneycontrol,SBI Consolidated December 2021 Net Interest In...,NaN
2,SBIN,2022-02-08,moneycontrol,SBI Standalone December 2021 Net Interest Inco...,NaN


## 1. Combined text column

In [2]:
df['text'] = df['text'].fillna('')
df['combined_text'] = (df['headline'] + ' ' + df['text']).str.strip()
# Truncate to 512 tokens-worth of characters for transformer compatibility
df['combined_text_512'] = df['combined_text'].str[:1500]
print('combined_text sample:', df['combined_text'].iloc[0][:200])

combined_text sample: What should investors do with SBI after Q3 results; buy, sell or hold?


## 2. VADER sentiment features

In [3]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'vaderSentiment', '-q'])
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Custom banking/finance terms to boost
BANKING_BOOSTS = {
    'npa': -1.5, 'fraud': -2.0, 'default': -1.5, 'downgrade': -1.8,
    'stressed': -1.2, 'recovery': +1.5, 'profit': +1.8, 'dividend': +1.5,
    'acquisition': +0.8, 'merger': +0.8, 'rally': +1.5, 'surge': +1.5,
    'crash': -2.0, 'collapse': -2.0, 'penalty': -1.5, 'fine': -1.0,
    'upgrade': +1.5, 'growth': +1.2, 'record': +1.0
}
sia.lexicon.update(BANKING_BOOSTS)

def vader_scores(text):
    s = sia.polarity_scores(str(text))
    return s['pos'], s['neg'], s['neu'], s['compound']

tqdm.pandas(desc='VADER')
df[['vader_pos','vader_neg','vader_neu','vader_compound']] = df['headline'].progress_apply(
    lambda t: pd.Series(vader_scores(t)))

print('VADER features added.')
df[['vader_pos','vader_neg','vader_compound']].describe()

VADER:   0%|          | 0/13649 [00:00<?, ?it/s]

VADER features added.


,vader_pos,vader_neg,vader_compound
count,13649.000000,13649.000000,13649.000000
mean,0.091794,0.097241,-0.005081
std,0.117151,0.126206,0.377197
min,0.000000,0.000000,-0.915300
25%,0.000000,0.000000,-0.273200
50%,0.000000,0.000000,0.000000
75%,0.168000,0.178000,0.273200
max,0.655000,0.648000,0.933700


## 3. Temporal features

In [4]:
df['day_of_week']  = df['date'].dt.dayofweek           # 0=Mon … 4=Fri
df['month']        = df['date'].dt.month
df['quarter']      = df['date'].dt.quarter
df['is_monday']    = (df['day_of_week'] == 0).astype(int)
df['is_friday']    = (df['day_of_week'] == 4).astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
df['is_qtr_end']   = df['date'].dt.is_quarter_end.astype(int)
print('Temporal features added.')

Temporal features added.


## 4. Fetch forward price labels from Yahoo Finance

In [5]:
TICKER_MAP = {
    'HDFCBANK'  : 'HDFCBANK.NS',
    'ICICIBANK' : 'ICICIBANK.NS',
    'SBIN'      : 'SBIN.NS',
    'AXISBANK'  : 'AXISBANK.NS',
    'KOTAKBANK' : 'KOTAKBANK.NS',
    'INDUSINDBK': 'INDUSINDBK.NS',
}

start = (df['date'].min() - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
end   = (df['date'].max() + pd.Timedelta(days=15)).strftime('%Y-%m-%d')

price_cache = {}
for tk, yf_sym in tqdm(TICKER_MAP.items(), desc='Fetching prices'):
    try:
        hist = yf.Ticker(yf_sym).history(start=start, end=end)
        if not hist.empty:
            hist = hist[['Close']].copy()
            hist.index = pd.to_datetime(hist.index).tz_localize(None).normalize()
            price_cache[tk] = hist
    except Exception as e:
        print(f'  {tk}: {e}')

print(f'\nPrice data fetched for: {list(price_cache.keys())}')

Fetching prices:   0%|          | 0/6 [00:00<?, ?it/s]


Price data fetched for: ['HDFCBANK', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK', 'INDUSINDBK']


In [6]:
def get_fwd_return(ticker, news_date, days=1):
    """Return forward percentage price change `days` trading days after news_date."""
    if ticker not in price_cache:
        return np.nan
    prices = price_cache[ticker]
    future = prices[prices.index > news_date].sort_index()
    if len(future) < days:
        return np.nan
    today_rows = prices[prices.index <= news_date].sort_index()
    if today_rows.empty:
        return np.nan
    p0  = float(today_rows.iloc[-1]['Close'])
    p_n = float(future.iloc[days - 1]['Close'])
    return (p_n - p0) / p0

tqdm.pandas(desc='1-day return')
df['fwd_ret_1d'] = df.progress_apply(lambda r: get_fwd_return(r['ticker'], r['date'], 1), axis=1)
tqdm.pandas(desc='5-day return')
df['fwd_ret_5d'] = df.progress_apply(lambda r: get_fwd_return(r['ticker'], r['date'], 5), axis=1)

print('Return coverage:', df['fwd_ret_1d'].notna().sum(), '/', len(df))

1-day return:   0%|          | 0/13649 [00:00<?, ?it/s]

5-day return:   0%|          | 0/13649 [00:00<?, ?it/s]

Return coverage: 13634 / 13649


## 5. Create binary labels

In [7]:
THRESHOLD = 0.003   # ±0.3% threshold — below = NEUTRAL

def label_movement(ret):
    if pd.isna(ret): return np.nan
    if ret >  THRESHOLD: return 'UP'
    if ret < -THRESHOLD: return 'DOWN'
    return 'NEUTRAL'

df['label_1d'] = df['fwd_ret_1d'].apply(label_movement)
df['label_5d'] = df['fwd_ret_5d'].apply(label_movement)

print('1-day label distribution:')
print(df['label_1d'].value_counts(dropna=False))
print('\n5-day label distribution:')
print(df['label_5d'].value_counts(dropna=False))

1-day label distribution:
label_1d
DOWN       5225
UP         5165
NEUTRAL    3244
NaN          15
Name: count, dtype: int64

5-day label distribution:
label_5d
UP         5908
DOWN       5826
NEUTRAL    1498
NaN         417
Name: count, dtype: int64


## 6. Save feature-enriched dataset

In [8]:
out_path = PROC / 'features.csv'
df.to_csv(out_path, index=False)
print(f'✅ Saved features.csv  shape={df.shape}  →  {out_path}')

# Also save the price cache for backtesting
with open(PROC / 'price_cache.pkl', 'wb') as f:
    pickle.dump(price_cache, f)
print('✅ Saved price_cache.pkl')

✅ Saved features.csv  shape=(13649, 22)  →  ..\data\processed\features.csv
✅ Saved price_cache.pkl
